# 演習7 解答編 ―― パイプラインを組む

## 発展課題1 の解答 ―― Read を2人にしたら

**何も変わりません。**

Read は 10ms、ボトルネックは Infer（60ms）です。Read 係はもともと大半の時間を
手待ちで過ごしています。手待ちの人を増やしても、手待ちが2人になるだけです。

```
Read   R.........R.........R.........      ← すでに '.' だらけ
Infer  IIIIIIIIIIIIIIIIIIIIIIIIIIIIII      ← ここが詰まっている
```

演習1の発展課題3と同じ答えです。

> **ボトルネックでない段を速くしても、全体は1ミリも速くならない。**

しかも悪いことに、**Read を増やすと順番が崩れます**（7-2-6 と同じ理由）。
**得は何もなく、副作用だけがあります。**

そして 7-2-5 で実際に見たとおり、上限B で止まっているところにスレッドを足すと、
切り替えの手間のぶん**遅くなります**。「とりあえず並列にしておく」は、たいてい損です。

## 発展課題2 の解答 ―― キューの容量を 1 にしたら

**7-2 の構成では、ほとんど変わりません。** 意外に思うかもしれません。

理由は演習6-2 で見たとおりです。容量が効くのは
**「ボトルネックの段が、次のデータを待って手待ちになる」ときだけ**でした。

7-2 では、そうなりません。

- Read は 10ms。Infer よりずっと速いので、**供給が途切れません**
- Infer は2人。片方が 95ms のフレームで詰まっていても、**もう片方が流し続けます**

つまり **並列化そのものが、でこぼこを吸収するクッションになっています。**

ただし、**Read 側にでこぼこがあれば話は変わります。** 次のセルで両方を測ります。
でこぼこは「待ち」だけで作り、**CPU 総量は動かしていません**。
上限B まで動いてしまうと、容量の効果と区別できなくなるためです。

In [ ]:
%%writefile ans07b.cpp
#include <iostream>
#include <iomanip>
#include <thread>
#include <vector>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

// ---- 演習5で組み立てたキュー（読み飛ばしてよい） ----
template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity) : capacity_(capacity) {}
    void push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_; });
        q_.push(v); lk.unlock(); can_pop_.notify_one();
    }
    T pop() {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });
        T v = q_.front(); q_.pop(); lk.unlock(); can_push_.notify_one();
        return v;
    }
private:
    std::queue<T> q_;
    std::size_t capacity_;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_, can_push_;
};

const int N = 30;
long calib = 0;
volatile long sink = 0;
long burn(long n) { long s = 0; for (long i = 0; i < n; i++) s += (i * 2654435761u) % 7; return s; }
void calibrate() {
    long n = 100000;
    for (;;) {
        auto t0 = steady_clock::now();
        sink += burn(n);
        auto us = duration_cast<microseconds>(steady_clock::now() - t0).count();
        if (us > 30000) { calib = n * 1000 / us; break; }
        n *= 2;
    }
}
void stage(int cpu_ms, int wait_ms) {           // 本編と同じ「計算 + 待ち」
    if (cpu_ms)  sink += burn(calib * cpu_ms);
    if (wait_ms) std::this_thread::sleep_for(milliseconds(wait_ms));
}

// Read の CPU は 10ms で固定。でこぼこは「待ち」だけで作る（CPU 総量を動かさないため）
bool read_jitter = false;
int JIT = 150;
void do_read(int i)  { read_jitter ? stage(10, (i % 5 == 4) ? JIT : 0) : stage(10, 0); }
void do_infer(int i) { stage(25, (i % 3) * 35); }      // 25 / 60 / 95ms
void do_show()       { stage(12, 18); }                // 30ms

int run(std::size_t cap) {
    BoundedQueue<int> q1(cap), q2(cap);
    auto t0 = steady_clock::now();
    std::thread reader([&] { for (int i = 0; i < N; i++) { do_read(i); q1.push(i); } });
    std::vector<std::thread> inferers;
    for (int k = 0; k < 2; k++)
        inferers.emplace_back([&, k] {
            for (int i = k; i < N; i += 2) { int f = q1.pop(); do_infer(f); q2.push(f); }
        });
    std::thread shower([&] { for (int i = 0; i < N; i++) { q2.pop(); do_show(); } });
    reader.join();
    for (auto& t : inferers) t.join();
    shower.join();
    return duration_cast<milliseconds>(steady_clock::now() - t0).count();
}

void table() {
    std::cout << "  capacity      FPS\n";
    for (std::size_t cap : {1, 2, 4, 8, 32}) {
        int ms = run(cap);
        std::cout << std::right << std::setw(8) << cap
                  << std::setw(9) << std::fixed << std::setprecision(1) << (1000.0 * N / ms) << "\n";
    }
}

int main() {
    calibrate();
    std::cout << N << "フレームを流し、FPS = " << N << "フレーム / かかった秒数 で出しています。\n\n";
    std::cout << "【A】Read の待ちはいつも 0ms（7-2 と同じ）\n";
    read_jitter = false; table();
    std::cout << "\n【B】Read だけ変える : 5回に1回だけ 150ms 待つ（CPU 総量は A と同じ）\n";
    read_jitter = true;  table();
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread -O2 ans07b.cpp -o ans07b && ./ans07b

```
【A】Read の待ちはいつも 0ms      【B】5回に1回だけ 150ms 待つ
  capacity      FPS                capacity      FPS
       1     23.2                       1     17.8
       2     25.0                       2     20.7
       4     25.1                       4     21.9
       8     23.6                       8     21.0
      32     23.5                      32     21.5
```

- **A … 容量を変えても差は出ません。** でこぼこがないので、吸収するものがありません
  （1割ほどの上下は実行ごとのぶれです）
- **B … 容量1だけがはっきり遅い**（17.8 FPS）。容量2で 20.7、容量4で 21.9。
  そこから先は変わりません

**B で容量1が遅い理由**は演習6-2 のとおりです。Read が 150ms 止まっているあいだ、
容量1では Infer に渡せる在庫が1枚しかありません。**Infer が手待ちになります。**
容量が4あれば、Read が速いフレームのうちに貯めた在庫でその 150ms を埋められます。

演習6-3 で書いた手順が、そのまま当てはまります。

> **1 から増やして、FPS が上がらなくなったところで止める。**

そして 7-3-2 に書いたとおり、**容量は2つの上限をどちらも動かしません。**
容量を増やして速くならないなら、**それは容量の問題ではありません。**
上限A か 上限B か、どちらで詰まっているのかを見に行ってください。

## 発展課題3 の解答 ―― 2段パイプラインにしたら

**遅くなります。** 計算だけで分かります。

```
3段 : Read 10 / Infer 60 / Show 30   ⇒ 一番遅い段は 60ms ⇒ 16.7 FPS
2段 : [Read+Infer] 70 / Show 30      ⇒ 一番遅い段は 70ms ⇒ 14.3 FPS
```

段をまとめると、まとめた段の時間は**足し算**になります。
一番遅い段がさらに遅くなるので、上限が下がります。

**そして上限B のほうは、1ミリも動きません。** 1フレームの CPU 総量は
`10 + 25 + 12 = 47ms` のままだからです。**段の切り方で動くのは、上限A だけ**です。

> **段は細かく切るほど上限A が上がる。上限B は動かない。**

理屈のうえでは「1段の時間ができるだけそろうように、できるだけ細かく切る」のが
最良です。ただし、細かく切れば切るほど次の2つが増えます。

- **キューの出し入れの回数**（1回あたりは軽いですが、回数が増えます）
- **レイテンシ**（段の数だけ待ち行列を通ります。7-2-6 で見たとおり）

そして何より、**切れないところは切れません。** Infer の中身が
「前処理 → 外部ハードウェアに投げる → 後処理」なら3つに切れますが、
外部ハードウェアの部分だけは、どう切っても短くなりません。

実際には「一番遅い段を、切れるところまで切る」だけで十分なことがほとんどです。

## 発展課題4 の解答 ―― 順番の崩れをどう直すか（案出し）

思いつく案を並べます。演習8で、このうちの1つを実際に作ります。

**案1 : 出口で並べ直す**

各フレームに番号を付けておき、最後の段で「次に出すべき番号」を持っておきます。
番号どおりに来なければ、来るまで手元にとっておきます。
**いちばん素直で、いちばんよく使われる方法**です。演習8で作ります。

**案2 : そもそも並列化しない**

順序が絶対に崩れてはいけない段は、1人でやらせる。速度はあきらめる。
**判断としては十分ありえます。** 7-2 で見たとおり、上限B に当たっていれば
そもそも2人にしても大して速くなっていません。**失うものが小さい**場合があります。

**案3 : 担当者ごとに出口を分けて、順番に回収する**

Infer 係A・B にそれぞれ専用の出口キューを持たせ、Read の時点で
「Aに0番、Bに1番、Aに2番...」と交互に配ります。
Show は A→B→A→B と順番に取りに行けば、順序は保たれます。

ただし、**片方が重いフレームを引くと、そちらが空くまで全体が止まります**。
配り方を固定した代償です。

**案4 : 順序をあきらめる**

表示だけなら、多少前後しても実害がない場合があります。
ただし「番号と結果の対応がずれる」形の崩れは、**表示でも許されません**。

どの案がよいかは「順序が崩れると何が困るのか」次第です。
**まず、それを決めてから選んでください。**